# Track 2 · 01 PROTEUS prepare — the warm-up data onto Drive, as the harness makes it

**One job.** Put the warm-up corpus on Drive in the form `track2_02_eegnet` trains on: the
harness's own 4-second windows, their three mental-command labels, and the people the
warm-up tests.

Self-contained: upload this one file.

---

## The task, from source

The warm-up now scores on **PROTEUS** (NEMAR `nm000290`, CC-BY-4.0): 10 participants,
14 sessions, 112 runs, 41 EEG channels at 500 Hz, cued kinesthetic motor imagery /
mental subtraction / word generation with Graz and BrainHero interfaces. Three classes,
so chance is 33.3%.

From `neuralbench/tasks/eeg/_motor_imagery_stream/config.yaml` (v0.4.0): one window per
cue, **1 to 5 s after the cue**, labels 0/1/2. The split is a subject-level
`SklearnSplit` (20% test, 20% valid, seed 33), **not** a fixed list like Dreyer's — the
test people are whatever the harness draws. We reproduce the same seed and ratios.

The data is read exactly as the harness reads it: through **NEMAR `nm000290`** (the
`Dreyer2026Proteus` study in neuralfetch 0.4.0), which downloads the BIDS EDF release.

## Preprocessing is the harness's, exactly

| step | value | at 500 Hz |
|---|---|---|
| notch | 50 and 60 Hz plus harmonics below Nyquist | 50, 100, 150, 200, 250 and 60, 120, 180, 240, 300 |
| band-pass | 0.1 – 75 Hz | both edges apply |
| resample | 120 Hz | 480 samples per window |
| scale | none (raw microvolts, no scaler, no clamp) | |
| baseline | none for this task | |

## Licence

PROTEUS is **CC-BY-4.0** (neuralfetch study licence). Cite Dreyer et al., *PROTEUS BCI
Bordeaux*, NEMAR nm000290 v1.0.0.

## Cost

About 112 small EDF runs from NEMAR, one session at a time to temporary disk and deleted
after. Windows are stored as float16 on Drive (µV scale, rounding error negligible).
Resumes: sessions already on Drive are skipped on a rerun.

## Setup

In [ ]:
ROOT = '/content/drive/MyDrive/Neurips26'

# The harness's settings. Do not tune these; matching them is the point.
SFREQ = 120.0
NOTCH = (50.0, 60.0)
BANDPASS = (0.1, 75.0)
CLAMP = None          # PROTEUS: no clamp, raw microvolts
START = 0.5           # window start relative to the cue, s
DURATION = 3.0        # window length, s (0.5-3.5 s post-cue, pre-feedback)

SUBJECTS = list(range(1, 11))
EVENT_CODES = {'mi': 0, 'sub': 1, 'word': 2}
TEST_RATIO = 0.2      # SklearnSplit in the stream config
VALID_RATIO = 0.2
SEED = 33

print('NOTEBOOK VERSION: track2_01 v2 PROTEUS')

In [ ]:
%pip install -q mne mne-bids pooch nemar-py

import importlib
for mod in ('mne', 'mne_bids', 'nemar', 'numpy', 'pandas'):
    print(f'{mod:12} {importlib.import_module(mod).__version__}')
print('mne_bids ok')

In [ ]:
import json, logging, shutil, time, warnings
warnings.filterwarnings('ignore')
from pathlib import Path
import numpy as np, mne
import pandas as pd
mne.set_log_level('ERROR')

if ROOT.startswith('/content/drive'):
    from google.colab import drive
    drive.mount('/content/drive')

OUT = Path(ROOT) / 'Track2' / 'dreyer2026proteus'
WIN = OUT / 'windows'
WIN.mkdir(parents=True, exist_ok=True)
TMP = Path('/content/nemar_tmp') if Path('/content').exists() else OUT.parent / '_nemar_tmp'
TMP.mkdir(parents=True, exist_ok=True)

# NEMAR nm000290 v1.0.0, as neuralfetch's Dreyer2026Proteus downloads it.
NEMAR_ID = 'nm000290'
NEMAR_VERSION = '1.0.0'
BIDS_ROOT = TMP / NEMAR_ID

from mne_bids import BIDSPath, get_entities_from_fname, read_raw_bids

## Step 1 — download, preprocess and window each session

Each session's runs are preprocessed together (the harness fits no scaler for PROTEUS,
so the raw microvolts pass through notch, band-pass and resample only), windows are cut
from each cue at 1–5 s, and the session's files are deleted from temporary disk before
the next one.

In [ ]:
def proteus_windows(raw, onsets, codes):
    """Harness preprocessing for one PROTEUS run, then 4-s windows from each cue."""
    raw = raw.copy().pick('eeg').load_data()
    nyq = raw.info['sfreq'] / 2
    freqs = [f for base in NOTCH for f in np.arange(base, min(nyq, 301), base)]
    if freqs:
        raw.notch_filter(freqs, phase='zero', filter_length='auto', verbose=False)
    lo, hi = BANDPASS
    raw.filter(lo, hi if hi < nyq else None, verbose=False)
    raw.resample(SFREQ, verbose=False)
    d = raw.get_data()                    # raw microvolts; no scaler, no clamp
    n_times = int(round(DURATION * SFREQ))
    idx = np.round((np.asarray(onsets) + START) * SFREQ).astype(int)
    keep = (idx >= 0) & (idx + n_times <= d.shape[1])
    X = np.stack([d[:, i:i + n_times] for i in idx[keep]]).astype('float32')
    return X, keep, list(raw.ch_names)


def do_session(s, ses):
    dest = WIN / f'sub-{s}_ses-{ses}.npz'
    if dest.exists():
        return 'kept', json.loads(str(np.load(dest)['info']))
    files = sorted(BIDS_ROOT.rglob(f'sub-{s}_ses-{ses}*_eeg.edf'))
    if not files:
        return 'skip', None
    Xs, ys, rs, per_run, ch_ref = [], [], [], {}, None
    for f in files:
        ents = get_entities_from_fname(f.name)
        bp = BIDSPath(root=BIDS_ROOT, subject=s, session=ses,
                      task=ents['task'], run=ents['run'], datatype='eeg',
                      suffix='eeg', extension='.edf')
        raw = read_raw_bids(bp, verbose='ERROR')
        events = pd.read_csv(bp.update(suffix='events', extension='.tsv').fpath, sep='\\t')
        cues = events[events.trial_type.isin(EVENT_CODES)].reset_index(drop=True)
        if cues.empty:
            continue   # baseline run: no cue events
        X, keep, ch = proteus_windows(raw, cues.onset.to_numpy(), cues.trial_type.to_numpy())
        if ch_ref is None:
            ch_ref = ch
        elif ch != ch_ref:
            raise RuntimeError(f'run {f.name}: channels differ from the first run')
        labels = cues.trial_type.map(EVENT_CODES).to_numpy()[keep]
        Xs.append(X.astype('float16')); ys.append(labels)
        rs.append(np.full(int(keep.sum()), bp.run))
        per_run[f'ses-{ses}_run-{int(bp.run):02d}_{ents["task"]}'] = int(keep.sum())
    if not Xs:
        return 'empty', None
    info = dict(subject=s, session=ses, channels=ch_ref, runs=per_run,
                n_windows=int(sum(per_run.values())), native_sfreq=500.0)
    np.savez(dest, X=np.concatenate(Xs), y=np.concatenate(ys).astype('int64'),
             run=np.concatenate(rs), info=json.dumps(info))
    return 'new', info


def ensure_download():
    """Download NEMAR nm000290 v1.0.0 to temporary disk once, idempotently."""
    marker = TMP / '.nm000290_done'
    if marker.exists() and any(BIDS_ROOT.rglob('*_eeg.edf')):
        return
    import nemar
    nemar.download(dataset=NEMAR_ID, tag=NEMAR_VERSION, target_dir=BIDS_ROOT, trust_existing=True)
    marker.write_text('ok')


ensure_download()
t0 = time.time()
done, failed, skipped = {}, {}, {}
import re as _re
_ent = _re.compile(r'sub-([A-Za-z0-9]+).*?ses-([A-Za-z0-9]+)')
sessions = sorted({(m.group(1), m.group(2)) for f in BIDS_ROOT.rglob('*_eeg.edf')
                   if (m := _ent.search(f.name))})
print(f'{len(sessions)} person-sessions found under {BIDS_ROOT}')
for i, (s, ses) in enumerate(sessions, 1):
    try:
        status, info = do_session(s, ses)
        if info is None:
            skipped[(s, ses)] = status
            print(f'  {i:2d}/{len(sessions)}  sub-{s} ses-{ses} {status}')
        else:
            done[(s, ses)] = info
            print(f'  {i:2d}/{len(sessions)}  sub-{s} ses-{ses} {status}  '
                  f'{info["n_windows"]} windows  {time.time() - t0:.0f}s')
    except Exception as e:
        failed[(s, ses)] = repr(e)
        print(f'  FAILED sub-{s} ses-{ses}: {e}')
print(f'\\n{len(done)} sessions written, {len(skipped)} skipped, {len(failed)} failed, '
      f'{time.time() - t0:.0f}s')
if failed:
    print('Rerun this cell: finished sessions are kept and only the failures are retried.')

## Step 2 — groups, manifest and checks

The stream config's `SklearnSplit` draws 20% of the 10 subjects to test and 20% to
valid with `random_state=33`. We reproduce that split here so our training runs use the
same train/valid/test people the harness would. Note this is the **local** split, not the
sealed protocol (which uses later sessions of these people plus 10 unseen people).

In [ ]:
from sklearn.model_selection import train_test_split

print(f"sessions written: {len(done)}")
if not done:
    edfs = list(BIDS_ROOT.rglob("*_eeg.edf"))
    print(f"no sessions written; found {len(edfs)} edf files under {BIDS_ROOT}")
    for f in edfs[:10]:
        print("  ", f.relative_to(BIDS_ROOT))
    print(f"windows on Drive: {len(list(WIN.glob("*.npz")))}")
    raise SystemExit("Step 1 wrote no sessions; run it again and read its output.")

ALL = sorted(set(s for (s, ses) in done))
train_subj, test_subj = train_test_split(ALL, test_size=TEST_RATIO, random_state=SEED, shuffle=True)
train_subj, val_subj = train_test_split(train_subj, test_size=VALID_RATIO / (1 - TEST_RATIO),
                                        random_state=SEED, shuffle=True)
group = {s: 'test' if s in test_subj else 'val' if s in val_subj else 'train' for s in ALL}

per_group, problems, recs, chans = {}, [], {}, set()
for f in sorted(WIN.glob('sub-*_ses-*.npz')):
    z = np.load(f)
    info = json.loads(str(z['info']))
    X, y = z['X'], z['y']
    g = group[info['subject']]
    recs[f.stem] = dict(info, group=g, file=f.name)
    chans.add(tuple(info['channels']))
    if X.shape[1:] != (41, int(DURATION * SFREQ)):
        problems.append(f'{f.name}: shape {X.shape}')
    if not np.isfinite(X.astype('float32')).all():
        problems.append(f'{f.name}: non-finite values')
    if set(np.unique(y)) - {0, 1, 2}:
        problems.append(f'{f.name}: labels {np.unique(y)}')
    p = per_group.setdefault(g, dict(sessions=0, runs=0, windows=0,
                                     mi=0, sub=0, word=0))
    p['sessions'] += 1; p['runs'] += len(info['runs']); p['windows'] += len(y)
    for k, c in EVENT_CODES.items():
        p[k] += int((y == c).sum())

for g in ('train', 'val', 'test'):
    p = per_group.get(g, {})
    print(f'  {g:5}  {p.get("sessions", 0):2d} sessions  {p.get("runs", 0):3d} runs  '
          f'{p.get("windows", 0):6,d} windows   mi {p.get("mi", 0):,} / sub {p.get("sub", 0):,} / word {p.get("word", 0):,}')
if len(chans) != 1:
    problems.append(f'{len(chans)} different channel lists across sessions')
total_runs = sum(p['runs'] for p in per_group.values())
print(f'\\n{len(recs)} sessions, {total_runs} runs (14 sessions written; 84 cued runs of 112 total (28 baseline runs have no cues))')
print(f'channels: {list(next(iter(chans))) if len(chans) == 1 else chans}')
print('checks:', 'all passed' if not problems else problems[:10])

(OUT / 'manifest.json').write_text(json.dumps(dict(
    source='PROTEUS (NEMAR nm000290 v1.0.0, Dreyer2026Proteus)', licence='CC-BY-4.0',
    preprocessing=dict(sfreq=SFREQ, notch=NOTCH, bandpass=BANDPASS, scaler=None,
                       baseline=None, clamp=None, stored_dtype='float16'),
    windows=dict(start=START, duration=DURATION,
                 labels={'mi': 0, 'sub': 1, 'word': 2}),
    split=dict(test=sorted(test_subj), val=sorted(val_subj), train=sorted(train_subj),
               val_note='ours: SklearnSplit seed 33, ratios 20/20 as the stream config'),
    per_group=per_group, recordings=recs, failed={str(k): v for k, v in failed.items()}), indent=1))
size = sum(f.stat().st_size for f in WIN.glob('*.npz')) / 1e6
print(f'\\nmanifest -> {OUT / "manifest.json"}   windows on Drive: {size:.0f} MB')
ready = len(recs) == 14 and total_runs == 84 and not failed and not problems
print('READY for track2_02' if ready else 'NOT READY: see the counts and checks above')